In [ ]:
# !pip cache purge

In [ ]:
!pip install numpy ipympl matplotlib 'torch>=2.11.0' 'torchaudio>=2.11.0' torchcodec misaki-fork[en] kokoro-onnx 

In [ ]:
!mkdir -p models && cd models && wget -nv -nc \
    https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin \
    https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx
!mkdir -p models/hifigan-v1-finetuned-pudge && cd models/hifigan-v1-finetuned-pudge && wget -nv -nc \
    https://huggingface.co/DaniilMiskevich/knn-vc-hifigan-v1-finetuned-pudge/resolve/main/g_00006000.pt \
    https://huggingface.co/DaniilMiskevich/knn-vc-hifigan-v1-finetuned-pudge/resolve/main/g_00016000.pt \
    https://huggingface.co/DaniilMiskevich/knn-vc-hifigan-v1-finetuned-pudge/resolve/main/g_00020000.pt \
    https://huggingface.co/DaniilMiskevich/knn-vc-hifigan-v1-finetuned-pudge/resolve/main/g_00022000.pt \
    https://huggingface.co/DaniilMiskevich/knn-vc-hifigan-v1-finetuned-pudge/resolve/main/g_00024000.pt \
    https://huggingface.co/DaniilMiskevich/knn-vc-hifigan-v1-finetuned-pudge/resolve/main/config.json

In [ ]:
!rm -rf target_voices/ && mkdir -p target_voices && cd target_voices && wget -nv \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/1/1c/Pud_spawn_03_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/2/29/Pud_spawn_09_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/8/87/Pud_battlebegins_01_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/5/59/Pud_move_06_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/9/98/Pud_ability_hook_03_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/9/94/Pud_ability_rot_10_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/3/39/Pud_ability_rot_13_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/0/09/Pud_ability_devour_15_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/2/2e/Pud_rival_07_ru.mp3

In [ ]:
import os
import sys
from IPython.display import display, Audio, HTML
import torch
import numpy as np
import json

from kokoro_onnx import Kokoro
from kokoro_onnx.config import SAMPLE_RATE as KOKORO_SR
from misaki import en, espeak

from torch import hub
import torchaudio
import librosa as lr
sys.path.append('./knn-vc')   
from hifigan.models import Generator as HiFiGAN
from hifigan.utils import AttrDict

# Вариант 6

### Задание

1. Используя CosyVoice3 (или любой другой по согласованию с преподавателем):

    - [x] озвучить текст

    - [x] oценить полученные результаты
      
1. Используя проект kNN-VC:

    - [x] сконвертировать свой голос в любой другой
  
    - выполнить дообучение вокодера (HiFi-GAN):

        - [x] найти тестовые данные (0.5-2 часа)
  
        - [x] сетап
  
        - [+/-] сам процесс обучения (4-8 часов)
     
    - [ ] оценить, насколько хорошо метод работает в условиях ограниченных ресурсов

In [ ]:
TEXT = '''
Speech synthesis is the artificial production of human speech. A computer system used for this purpose is called a speech synthesizer, and can be implemented in software or hardware products. A text-to-speech (TTS) system converts normal language text into speech; other systems render symbolic linguistic representations like phonetic transcriptions into speech. The reverse process is speech recognition.
'''

In [ ]:
KNN_VC_SR = 16000
def load_for_knn_vc(path):
    voice, sr = torchaudio.load(path)
    if sr != KNN_VC_SR: voice = torchaudio.functional.resample(voice, orig_freq=sr, new_freq=KNN_VC_SR)
    if voice.shape[0] > 1: voice = torch.mean(voice, dim=0, keepdim=True)
    return voice

orig_voices = list(map(load_for_knn_vc, ("./voice_by_alexei_braichuk.wav", "./borsch.wav", "laugh_from_freesound.mp3")))
target_voices = [load_for_knn_vc(f"./target_voices/{p}") for p in os.listdir("./target_voices")]

display(HTML("Original"))
for orig_voice in orig_voices: display(Audio(orig_voice, rate=KNN_VC_SR))
    
display(HTML("Target Voice"))
display(Audio(target_voices[1], rate=KNN_VC_SR))

### TTS

In [ ]:
g2p = en.G2P(trf=False, fallback=espeak.EspeakFallback(british=False))

In [ ]:
kokoro = Kokoro("./models/kokoro-v1.0.fp16-gpu.onnx", "./models/voices-v1.0.bin")

In [ ]:
phonemes_paragraphs = [g2p(t)[0] for t in TEXT.split("\n\n")]
for pp in phonemes_paragraphs:
    print(pp)

voiced = []

for voice in [ 
    # english-native
    "af_sky", "am_liam", "am_puck", "am_santa",
    # accents
    "hm_omega",
]:
    paragraphs = []
    for phonemes in phonemes_paragraphs:
        paragraph, _ = kokoro.create(phonemes, is_phonemes=True, voice=voice, speed=1.0)
        paragraphs.append(paragraph)
    v = np.concat([np.pad(p, int(0.3*KOKORO_SR)) for p in paragraphs])
    if paragraphs: voiced.append(v)
    display(Audio(v, rate=KOKORO_SR))


### VC

In [ ]:
knn_vc = hub.load(
    "bshall/knn-vc", "knn_vc",
    pretrained=True, prematched=True,
    trust_repo=True,
    device=torch.device(torch.cuda.is_available() and "cuda" or "cpu"),
)
hifigan = knn_vc.hifigan

matching_seq = knn_vc.get_matching_set(target_voices)

In [ ]:
CHECKPOINT = 6_000

with open('./models/hifigan-v1-finetuned-pudge/config.json') as f:
    h = AttrDict(json.load(f))
state_dict = torch.load(f'./models/hifigan-v1-finetuned-pudge/g_{str(CHECKPOINT).rjust(8, '0')}.pt', map_location='cpu')

hifigan_finetuned = HiFiGAN(h)
hifigan_finetuned.load_state_dict(state_dict['generator'])
hifigan_finetuned.eval()
hifigan_finetuned.remove_weight_norm()

hifigan_finetuned.to(knn_vc.device)
def get_matching_set_finetuned(target_voices):
    print("Using overrided `get_matching_set()`...")
    seqs = list(map(knn_vc.get_features, target_voices))
    return torch.cat(seqs, dim=0)

matching_seq_finetuned = get_matching_set_finetuned(target_voices)

In [ ]:
query_seqs = list(map(knn_vc.get_features, orig_voices))

def convert_voices(matching_seq, *, use_finetuned_hifigan=False):
    knn_vc.hifigan = hifigan_finetuned if use_finetuned_hifigan else hifigan
    for query_seq in query_seqs: 
        converted = knn_vc.match(query_seq, matching_seq, topk=4)
        display(Audio(converted, rate=KNN_VC_SR))

In [ ]:
convert_voices(matching_seq)

In [ ]:
convert_voices(matching_seq_finetuned, use_finetuned_hifigan=True)